# 08 - Weights & Biases Experiment Tracking

This notebook shows a compact end-to-end ML run tracked with **Weights & Biases (W&B)**.

## What it demonstrates
- run initialization
- configuration logging
- training / validation metric logging
- confusion matrix logging
- artifact logging
- online vs offline execution modes

The training example uses a small PyTorch classifier on FashionMNIST so the tracking workflow is the star of the show.


In [ ]:
# Colab setup
%pip -q install -U torch torchvision wandb scikit-learn matplotlib pandas


In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report

import wandb

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## W&B setup

In Colab you have two practical options:

- **online mode**: run `wandb.login()` and log to your real account
- **offline mode**: write the run locally and sync later

For a class demo, offline mode is often safer if you do not want to expose credentials on screen.


In [ ]:
# Choose ONE of these approaches.

# Option A: online mode
# wandb.login()

# Option B: offline mode
os.environ["WANDB_MODE"] = "offline"


In [ ]:
config = {
    "batch_size": 128,
    "epochs": 6,
    "learning_rate": 1e-3,
    "hidden_dim": 256,
    "dropout": 0.3,
    "dataset_limit_train": 12000,
    "dataset_limit_test": 3000,
}

run = wandb.init(
    project="cmpe258-advanced-nn",
    name="fashionmnist-wandb-demo",
    config=config,
)


## Dataset and model


In [ ]:
transform = transforms.ToTensor()
train_full = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

train_subset = torch.utils.data.Subset(train_full, list(range(config["dataset_limit_train"])))
test_subset = torch.utils.data.Subset(test_full, list(range(config["dataset_limit_test"])))

train_len = 10000
val_len = len(train_subset) - train_len
train_set, val_set = random_split(train_subset, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

class WandbMLP(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.net(x)

model = WandbMLP(config["hidden_dim"], config["dropout"]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
criterion = nn.CrossEntropyLoss()


In [ ]:
wandb.watch(model, log="all", log_freq=100)


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    losses = []
    all_preds = []
    all_targets = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y.cpu().numpy().tolist())

    return np.mean(losses), correct / total, all_targets, all_preds


## Training loop with W&B logging


In [ ]:
best_state = None
best_val_acc = -1.0

for epoch in range(config["epochs"]):
    model.train()
    batch_losses = []

    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())

        if step % 25 == 0:
            wandb.log(
                {
                    "train/batch_loss": loss.item(),
                    "train/epoch": epoch,
                    "train/step": epoch * len(train_loader) + step,
                }
            )

    train_loss = float(np.mean(batch_losses))
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    wandb.log(
        {
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(f"epoch={epoch+1} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "best_model.pt")


In [ ]:
model.load_state_dict(best_state)
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)

wandb.log(
    {
        "test/loss": test_loss,
        "test/accuracy": test_acc,
    }
)

print("test_acc:", test_acc)


## Confusion matrix and artifact logging


In [ ]:
class_names = train_full.classes

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true,
        preds=y_pred,
        class_names=class_names,
    )
})

artifact = wandb.Artifact("fashionmnist-best-model", type="model")
artifact.add_file("best_model.pt")
wandb.log_artifact(artifact)


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

pd.DataFrame({
    "metric": ["best_val_acc", "test_loss", "test_acc"],
    "value": [best_val_acc, test_loss, test_acc],
})


In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
run.finish()
print("Finished W&B run. In offline mode, sync later with:")
print("wandb sync wandb/offline-run-*")


## What to say in the video

- show the config dictionary first
- explain `wandb.init(...)`
- point out `wandb.watch(model)` for gradients/parameter tracking
- explain metric logging inside the training loop
- show the confusion matrix and artifact logging
- mention online vs offline mode for safe classroom demos

That is enough to satisfy the W&B requirement without turning the notebook into a ceremonial tribute to dashboards.
